In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pickle

In [ ]:
# load the data
train = pd.read_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/data/train_cleaned.csv')
test = pd.read_csv('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/data/test_cleaned.csv')


In [4]:
train.head()

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales
0,row_00000,PRD-PRFP9S,14.252,Low Fat,0.0271,frozen foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket,1764.98
1,row_00001,PRD-PXXK71,7.698,Low Fat,0.0720,health and hygiene,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket,342.13
2,row_00002,PRD-V5MOIJ,14.264,Regular,0.0421,canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,378.85
3,row_00003,PRD-UN5Z3J,12.647,Regular,0.0449,soft drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,5595.72
4,row_00004,PRD-6RDQYB,10.338,Regular,0.0120,meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket,2375.36


In [6]:
# Separate features and target
X_train = train.drop(['id', 'total_sales'], axis=1)
y_train = train['total_sales']
X_test = test.drop(['id'], axis=1)

print("Training features shape:", X_train.shape)
print("Target shape:", y_train.shape)
print("Test features shape:", X_test.shape)

Training features shape: (6818, 11)
Target shape: (6818,)
Test features shape: (1705, 11)


In [7]:
# identify column types
# Numerical columns
numerical_cols = ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years']

# Categorical columns
categorical_cols = ['fat_content', 'product_category', 'store_size', 'store_location_tier', 'store_format']

# Drop ID columns (product_code, store_code - not useful for prediction)
X_train = X_train.drop(['product_code', 'store_code'], axis=1)
X_test = X_test.drop(['product_code', 'store_code'], axis=1)

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years']
Categorical columns: ['fat_content', 'product_category', 'store_size', 'store_location_tier', 'store_format']


In [8]:
# create preprocessing Pipeline
# Define preprocessing for numerical columns
numerical_transformer = StandardScaler()

# Define preprocessing for categorical columns
categorical_transformer = OneHotEncoder(drop='first', sparse_output=False)

# Combine into one preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

print("Preprocessor created successfully!")


Preprocessor created successfully!


In [9]:
# fit and transfor data
# Fit on training data and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

# Get feature names after transformation
feature_names = numerical_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))
print(f"\nTotal features after encoding: {len(feature_names)}")
print("First 20 features:", feature_names[:20])

Processed training shape: (6818, 27)
Processed test shape: (1705, 27)

Total features after encoding: 27
First 20 features: ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years', 'fat_content_Regular', 'product_category_breads', 'product_category_breakfast', 'product_category_canned', 'product_category_dairy', 'product_category_frozen foods', 'product_category_fruits and vegetables', 'product_category_hard drinks', 'product_category_health and hygiene', 'product_category_household', 'product_category_meat', 'product_category_others', 'product_category_seafood', 'product_category_snack foods', 'product_category_soft drinks', 'product_category_starchy foods']


In [10]:
# calculate the VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Create DataFrame for VIF calculation
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = feature_names
vif_data["VIF"] = [variance_inflation_factor(X_train_processed, i) for i in range(X_train_processed.shape[1])]

# Sort by VIF
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\nVariance Inflation Factor (VIF):")
print(vif_data.head(15))

# Flag high VIF features (> 5)
high_vif = vif_data[vif_data['VIF'] > 5]
if len(high_vif) > 0:
    print(f"\n⚠️  High multicollinearity detected in {len(high_vif)} features")
    print(high_vif)
else:
    print("\n✅ No high multicollinearity detected")


Variance Inflation Factor (VIF):
                                   Feature         VIF
20                       store_size_Medium  103.207833
21                        store_size_Small   83.537979
3                          store_age_years   49.419901
23              store_location_tier_Tier_3   32.930124
25       store_format_Standard Supermarket   29.779918
24       store_format_Flagship Hypermarket   14.082301
26                 store_format_Superstore   10.717740
22              store_location_tier_Tier_2    7.618102
10  product_category_fruits and vegetables    2.529435
17            product_category_snack foods    2.442206
13              product_category_household    2.314638
9            product_category_frozen foods    2.085486
8                   product_category_dairy    1.925739
7                  product_category_canned    1.854535
12     product_category_health and hygiene    1.769346

⚠️  High multicollinearity detected in 8 features
                              Featu

In [11]:
# feature importance check
# Insight: product_price is strongest predictor
# Create interaction: price × store_age (older stores might have different sales patterns)

X_train_interaction = X_train.copy()
X_test_interaction = X_test.copy()

X_train_interaction['price_x_store_age'] = X_train_interaction['product_price'] * X_train_interaction['store_age_years']
X_test_interaction['price_x_store_age'] = X_test_interaction['product_price'] * X_test_interaction['store_age_years']

print("Interaction feature created: price_x_store_age")
print(X_train_interaction[['product_price', 'store_age_years', 'price_x_store_age']].head())

Interaction feature created: price_x_store_age
   product_price  store_age_years  price_x_store_age
0          81.37               45            3661.65
1          42.05               35            1471.75
2          41.35               33            1364.55
3         174.35               47            8194.45
4         203.06               28            5685.68


In [12]:
# create the final pipeline with interaction features
# If you want to include interaction features, update the preprocessor
numerical_cols_updated = numerical_cols + ['price_x_store_age']

preprocessor_final = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols_updated),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ])

# Fit final preprocessor
X_train_final = preprocessor_final.fit_transform(X_train_interaction)
X_test_final = preprocessor_final.transform(X_test_interaction)

print("Final processed training shape:", X_train_final.shape)
print("Interaction features included ✓")

Final processed training shape: (6818, 28)
Interaction features included ✓


In [15]:
# save the preprocessor for the building phase
# Save the fitted preprocessor to use in model building phase
with open('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/models/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor_final, f)

print("Preprocessor saved to models/preprocessor.pkl")

# Also save feature names
feature_names_final = numerical_cols_updated + list(preprocessor_final.named_transformers_['cat'].get_feature_names_out(categorical_cols))
with open('/Users/EXOTISCH TECH/Desktop/dsn-bootcamp-hackathon/models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names_final, f)

print("Feature names saved")

Preprocessor saved to models/preprocessor.pkl
Feature names saved


In [13]:
print("\n=== FEATURE ENGINEERING SUMMARY ===")
print(f"Original features: {X_train.shape[1]}")
print(f"After encoding: {X_train_final.shape[1]}")
print(f"Target variable shape: {y_train.shape}")
print("\nReady for model building! ✓")


=== FEATURE ENGINEERING SUMMARY ===
Original features: 9
After encoding: 28
Target variable shape: (6818,)

Ready for model building! ✓


In [14]:
print("\n=== MULTICOLLINEARITY STRATEGY ===")
print("High VIF detected in categorical features — this is normal with one-hot encoding.")
print("\nStrategy:")
print("- Tree-based models (Random Forest, XGBoost): NOT affected ✓")
print("- Linear Regression: Will use Ridge regression to handle multicollinearity ✓")
print("\nNo features dropped. All information retained.")


=== MULTICOLLINEARITY STRATEGY ===
High VIF detected in categorical features — this is normal with one-hot encoding.

Strategy:
- Tree-based models (Random Forest, XGBoost): NOT affected ✓
- Linear Regression: Will use Ridge regression to handle multicollinearity ✓

No features dropped. All information retained.
